# Modul B · Kapitel 4 — Quantisierung verstehen

> 🛠️ **Workshop-Version:** Bearbeite die zwei markierten Aufgaben.

**Lernziel:** Du kannst erklären, wie weniger Bits Speicher sparen und Genauigkeit kosten.

Dieses Notebook folgt einem kurzen Pfad: Begriff verstehen → Rechnung oder
Messung durchführen → Ergebnis für eine Deployment-Entscheidung nutzen.
Programmiert werden nur zwei Kernstellen: einfache und blockweise Quantisierung. Hilfs- und
Visualisierungscode ist bewusst vorgegeben.


## 0 · Setup

Die nächsten Zellen laden PyTorch, ein kleines Modell und die Hardwaredaten. Ein Modellserver ist nicht nötig.


In [ ]:
import sys
from pathlib import Path

# helfer.py liegt neben dem Notebook. Der Suchlauf findet es auch, wenn das
# Arbeitsverzeichnis woanders liegt — etwa in Colab.
for kandidat in [Path.cwd(), Path.cwd() / "04_deployment", *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import warnings

# tqdm meldet in Jupyter ohne ipywidgets eine Warnung, die nichts mit dem Thema zu tun hat.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import matplotlib
    import torch
    import transformers
except ImportError:
    %pip install -q matplotlib pandas torch transformers openai tiktoken
    import matplotlib
    import torch
    import transformers

import matplotlib.pyplot as plt
import numpy as np

import helfer
from helfer import GB, lade_daten, zeige_tabelle

transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
torch.set_grad_enabled(False)          # hier wird nichts trainiert

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 10,
})

print(f"torch {torch.__version__}   transformers {transformers.__version__}")
print(f"GB = {GB:,} Byte".replace(",", "."))
print("Setup fertig ✔")


In [ ]:
# ▶️ Die Datendateien für den letzten Abschnitt
MODELLE = lade_daten("model_cards")["modelle"]
HARDWARE = lade_daten("hardware")["karten"]

MODELL_NACH_NAME = {m["name"]: m for m in MODELLE}
KARTE_NACH_NAME = {k["name"]: k for k in HARDWARE}

print(f"{len(MODELLE)} Model Cards, {len(HARDWARE)} Karten")


In [ ]:
# ▶️ Das Modell, an dem gemessen wird. Der erste Aufruf lädt es herunter.
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELLNAME = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(MODELLNAME)
sprachmodell = AutoModelForCausalLM.from_pretrained(MODELLNAME, dtype=torch.float32)
sprachmodell.eval()
for p in sprachmodell.parameters():
    p.requires_grad_(False)          # hier wird nichts trainiert

PARAMETER_GESAMT = sum(p.numel() for p in sprachmodell.parameters())
einbettung = sprachmodell.model.embed_tokens.weight.numel()

print(f"{MODELLNAME}")
print(f"  Parameter insgesamt: {PARAMETER_GESAMT:,}".replace(",", "."))
print(f"  davon Einbettung:    {einbettung:,}".replace(",", "."))
print(f"  Schichten:           {sprachmodell.config.num_hidden_layers}")
print(f"  hidden_size:         {sprachmodell.config.hidden_size}")
print(f"  Format im Speicher:  {next(sprachmodell.parameters()).dtype}")


## 1 · Warum Präzision zählt

Weniger Bits pro Gewicht bedeuten weniger Speicher und meist mehr Geschwindigkeit. Dafür werden viele ursprüngliche Werte auf dasselbe Raster abgebildet.


In [ ]:
# ▶️ Llama 3.1 8B gegen jede Karte, in drei Präzisionen
BITS = {"FP16": 16, "INT8": 8, "INT4": 4}
BEISPIEL = MODELL_NACH_NAME["meta-llama/Llama-3.1-8B-Instruct"]


def speicher_gewichte(parameter, bits):
    """Speicher für die Gewichte in GB (GB = 10^9 Byte)."""
    return parameter * bits / 8 / GB


bedarf = {p: speicher_gewichte(BEISPIEL["parameter_gesamt"], b) for p, b in BITS.items()}

print(f"{BEISPIEL['name']}  —  "
      f"{BEISPIEL['parameter_gesamt'] / 1e9:.2f} Mrd. Parameter")
print("   ".join(f"{p}: {g:.2f} GB" for p, g in bedarf.items()))
print()

zeige_tabelle([{
    "Karte": k["name"],
    "VRAM (GB)": k["vram_gb"],
    **{p: ("passt" if g <= k["vram_gb"] else f"{g - k['vram_gb']:.2f} GB zu viel")
       for p, g in bedarf.items()},
} for k in HARDWARE])


## 2 · Ein Gewicht ist nur eine Zahl

FP32 speichert feine Abstufungen. INT8 oder INT4 speichern diskrete Stufen plus eine Skalierung, mit der Werte näherungsweise rekonstruiert werden.


In [ ]:
# ▶️ Eine echte Gewichtsmatrix
GEWICHT = sprachmodell.model.layers[10].mlp.down_proj.weight.numpy().astype(np.float32)

print(f"Form:               {GEWICHT.shape[0]} x {GEWICHT.shape[1]}   "
      f"({GEWICHT.size:,} Werte)".replace(",", "."))
print(f"Mittelwert:         {GEWICHT.mean():+.5f}")
print(f"Standardabweichung: {GEWICHT.std():.5f}")
print(f"kleinster Wert:     {GEWICHT.min():+.5f}")
print(f"größter Wert:       {GEWICHT.max():+.5f}")
print(f"größter Betrag:     {np.abs(GEWICHT).max():.5f}")
print()
for anteil in (50, 90, 99, 99.9, 100):
    print(f"  {anteil:>5} % der Beträge liegen unter "
          f"{np.percentile(np.abs(GEWICHT), anteil):.4f}")
print()
ueber_eins = int((np.abs(GEWICHT) > 1).sum())
print(f"Werte mit Betrag über 1: {ueber_eins} von {GEWICHT.size:,} "
      f"({ueber_eins / GEWICHT.size * 100:.3f} %)".replace(",", "."))


In [ ]:
# ▶️ Die Verteilung als Bild. Die y-Achse ist logarithmisch, sonst sieht man die Ausreißer nicht.
fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4.0))

links.hist(GEWICHT.reshape(-1), bins=200, color=BLAU)
links.set_yscale("log")
links.set_xlabel("Gewicht")
links.set_ylabel("Anzahl (logarithmisch)")
links.set_title("Alle Werte der Matrix")

grenze = float(np.percentile(np.abs(GEWICHT), 99.9))
for wert, farbe, text in ((grenze, TEAL, "99,9 % liegen darunter"),
                          (float(np.abs(GEWICHT).max()), ORANGE, "größter Betrag")):
    links.axvline(wert, color=farbe, linestyle="--", linewidth=1.2)
    links.axvline(-wert, color=farbe, linestyle="--", linewidth=1.2)
    links.text(wert, links.get_ylim()[1] * 0.3, f"  {text}", color=farbe, fontsize=9)

kern = GEWICHT.reshape(-1)[np.abs(GEWICHT.reshape(-1)) <= grenze]
rechts.hist(kern, bins=200, color=TEAL)
rechts.set_xlabel("Gewicht")
rechts.set_ylabel("Anzahl")
rechts.set_title("Derselbe Kern ohne die äußersten 0,1 %")

plt.tight_layout()
plt.show()


In [ ]:
# ▶️ Wertebereich gegen Auflösung, gemessen
print(f"{'Format':<10} {'Bits':>5} {'größte Zahl':>14} {'kleinste normale':>18} {'Auflösung':>12}")
for name, typ in (("FP32", torch.float32), ("FP16", torch.float16), ("BF16", torch.bfloat16)):
    f = torch.finfo(typ)
    print(f"{name:<10} {f.bits:>5} {f.max:>14.4g} {f.tiny:>18.4g} {f.eps:>12.4g}")

print()
proben = torch.tensor([1.5e5, 1.0e-8, 0.017345, 1.618034])
print(f"{'Wert':>14} {'in FP16':>18} {'in BF16':>18}")
for wert, in_fp16, in_bf16 in zip(proben, proben.half().float(), proben.bfloat16().float()):
    print(f"{wert:>14.6g} {in_fp16:>18.8g} {in_bf16:>18.8g}")

print()
print("Die ersten beiden Werte liegen außerhalb des FP16-Bereichs: der eine wird unendlich,")
print("der andere null. Die letzten beiden liegen drin — dort ist FP16 das genauere Format.")
print()

# Gegenprobe: Übersteht die Gewichtsmatrix den Weg durch BF16 unverändert?
als_tensor = sprachmodell.model.layers[10].mlp.down_proj.weight
print(f"Matrix unverändert nach BF16 und zurück: "
      f"{torch.equal(als_tensor, als_tensor.bfloat16().float())}")
print("Das Modell rechnet hier in FP32 — die Werte im Checkpoint sind BF16.")


## 3 · Absmax-Quantisierung

Wir skalieren den größten Betrag auf den größten darstellbaren Integer. Danach wird gerundet; beim Dequantisieren bleibt ein kleiner Fehler.


### 🛠️ Aufgabe 1 — Quantisieren und rekonstruieren

Implementiere die Absmax-Quantisierung. Bestimme die Skalierung, runde auf das Integer-Raster und rekonstruiere die Werte.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def quantisiere_absmax(x, bits=8):
    """Symmetrische Absmax-Quantisierung. Rückgabe: ganze Zahlen und Skala."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 1: quantisiere_absmax() implementieren")


def dequantisiere(q, skala):
    """Rechnet die ganzen Zahlen mit der Skala zurück in Fließkommawerte."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 1: dequantisiere() implementieren")


In [ ]:
# ✅ Selbsttest
probe = np.array([-1.0, -0.5, 0.0, 0.25, 1.0], dtype=np.float32)
q, skala = quantisiere_absmax(probe, 8)
zurueck = dequantisiere(q, skala)

assert q.dtype == np.int8, f"Die ganzen Zahlen sollen np.int8 sein, sind aber {q.dtype}"
assert q.min() >= -127 and q.max() <= 127, "8 Bit symmetrisch heißt -127 bis 127"
assert q.tolist() == [-127, -64, 0, 32, 127], f"q = {q.tolist()}"
assert abs(skala - 1 / 127) < 1e-9, f"Skala ist max(|x|)/127, nicht {skala}"
assert q[2] == 0 and zurueck[2] == 0.0, "Null bleibt Null — das ist der Sinn von symmetrisch"
assert np.abs(zurueck - probe).max() <= skala / 2 + 1e-9, \
    "Kein Wert darf weiter als eine halbe Schrittweite wandern"

q4, skala4 = quantisiere_absmax(probe, 4)
assert q4.min() >= -7 and q4.max() <= 7, f"4 Bit heißt -7 bis 7, nicht {q4.tolist()}"
assert abs(skala4 - 1 / 7) < 1e-9, "Bei 4 Bit ist die Schrittweite 18-mal so groß"

q0, skala0 = quantisiere_absmax(np.zeros(8, dtype=np.float32), 8)
assert skala0 == 1.0 and not np.any(q0), "Nur Nullen: Skala 1.0, keine Division durch null"

print("✅ Aufgabe 1 gelöst")
print()
print(f"{'x':>9} {'q (8 Bit)':>10} {'x̂ (8 Bit)':>11} {'q (4 Bit)':>10} {'x̂ (4 Bit)':>11}")
zurueck4 = dequantisiere(q4, skala4)
for i, wert in enumerate(probe):
    print(f"{wert:>9.4f} {q[i]:>10d} {zurueck[i]:>11.4f} {q4[i]:>10d} {zurueck4[i]:>11.4f}")
print()
print(f"Schrittweite 8 Bit: {skala:.6f}     Schrittweite 4 Bit: {skala4:.6f}")


In [ ]:
# ▶️ Dieselben Funktionen auf der echten Matrix
for bits in (8, 4):
    q, s = quantisiere_absmax(GEWICHT, bits)
    stufen = len(np.unique(q))
    moeglich = 2 ** bits - 1
    print(f"{bits} Bit:  Schrittweite {s:.5f}   "
          f"genutzte Stufen {stufen:>3} von {moeglich:>3}")

print()
print("Bei 4 Bit spannt der eine Ausreißer bei 4,72 ein Gitter mit der Schrittweite 0,67 auf.")
print("Der Kern der Verteilung liegt unter 0,75 — dort bleiben drei Gitterlinien übrig:")
print("-0,67, 0 und +0,67.")


## 4 · Fehler messen

Der Quantisierungsfehler ist die Differenz zwischen Original und Rekonstruktion. MAE und maximaler Fehler machen ihn sichtbar.


In [ ]:
# Vorgegebene Hilfsfunktion
def quantisierungsfehler(original, rekonstruiert):
    """Mittlerer und größter absoluter Fehler sowie der relative Fehler."""
    original = np.asarray(original, dtype=np.float64)
    abweichung = np.abs(original - np.asarray(rekonstruiert, dtype=np.float64))
    betrag = float(np.abs(original).sum())
    return {
        "mittel": float(abweichung.mean()),
        "max": float(abweichung.max()),
        "relativ": float(abweichung.sum() / betrag) if betrag > 0 else 0.0,
    }


## 5 · Blockweise quantisieren

Eine einzige Skalierung ist empfindlich für Ausreißer. Kleine Blöcke erhalten eigene Skalierungen und nutzen das Zahlenraster besser.


### 🛠️ Aufgabe 2 — Blockweise quantisieren

Quantisiere jeden Block mit einer eigenen Skalierung und setze die rekonstruierten Blöcke wieder zusammen.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def quantisiere_blockweise(x, bits=4, blockgroesse=64):
    """Absmax-Quantisierung mit einer eigenen Skala je Block."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 2: quantisiere_blockweise() implementieren")


In [ ]:
# ✅ Selbsttest
probe = np.full(128, 0.1, dtype=np.float32)
probe[70] = 8.0                                   # ein Ausreißer im zweiten Block

q, skalen = quantisiere_blockweise(probe, bits=4, blockgroesse=64)
assert q.shape == (2, 64), f"q soll (2, 64) sein, ist {q.shape}"
assert skalen.shape == (2, 1), f"skalen soll (2, 1) sein, ist {skalen.shape}"
assert q.dtype == np.int8, f"ganze Zahlen als np.int8, nicht {q.dtype}"

wieder = dequantisiere(q, skalen).reshape(probe.shape)
assert np.allclose(wieder[:64], 0.1), "Der erste Block kennt den Ausreißer nicht"
assert abs(wieder[70] - 8.0) < 1e-6, "Der Ausreißer selbst spannt seinen Block auf"
assert wieder[65] == 0.0, "Seine Nachbarn im selben Block fallen auf die Null-Linie"

# Auf der echten Matrix: Blöcke schlagen die ganze Matrix, kleinere Blöcke schlagen größere.
ganz = quantisierungsfehler(GEWICHT, dequantisiere(*quantisiere_absmax(GEWICHT, 4)))
b128 = quantisierungsfehler(GEWICHT, dequantisiere(
    *quantisiere_blockweise(GEWICHT, 4, 128)).reshape(GEWICHT.shape))
b32 = quantisierungsfehler(GEWICHT, dequantisiere(
    *quantisiere_blockweise(GEWICHT, 4, 32)).reshape(GEWICHT.shape))
assert b128["relativ"] < ganz["relativ"] / 5, "Blöcke müssen deutlich besser sein"
assert b32["relativ"] < b128["relativ"], "Kleinere Blöcke, kleinerer Fehler"

print("✅ Aufgabe 2 gelöst")
print()
print("Der Testvektor: 128 Werte à 0,1, an Stelle 70 eine 8,0")
print(f"  Block 0 (ohne Ausreißer): {wieder[:4]}  …")
print(f"  Block 1 (mit Ausreißer):  {wieder[64:68]}  …   Stelle 70: {wieder[70]}")
print()
print(f"Die echte Matrix in 4 Bit, relativer Fehler:")
print(f"  ganze Matrix:   {ganz['relativ'] * 100:6.2f} %")
print(f"  Blöcke à 128:   {b128['relativ'] * 100:6.2f} %")
print(f"  Blöcke à  32:   {b32['relativ'] * 100:6.2f} %")


In [ ]:
# ▶️ Der Fehler über die Blockgröße, für 8 und 4 Bit
BLOCKGROESSEN = [16, 32, 64, 128, 256, 512, 1024, 4096]

kurven, ohne_bloecke = {}, {}
for bits in (8, 4):
    kurven[bits] = [
        quantisierungsfehler(
            GEWICHT,
            dequantisiere(*quantisiere_blockweise(GEWICHT, bits, bg)).reshape(GEWICHT.shape),
        )["relativ"] * 100
        for bg in BLOCKGROESSEN
    ]
    ohne_bloecke[bits] = quantisierungsfehler(
        GEWICHT, dequantisiere(*quantisiere_absmax(GEWICHT, bits)))["relativ"] * 100

fig, achse = plt.subplots(figsize=(9.5, 4.4))
for bits, farbe in ((8, TEAL), (4, ORANGE)):
    achse.plot(BLOCKGROESSEN, kurven[bits], marker="o", color=farbe, label=f"{bits} Bit")
    achse.axhline(ohne_bloecke[bits], color=farbe, linestyle="--", linewidth=1.0)
    achse.text(BLOCKGROESSEN[-1], ohne_bloecke[bits] * 1.06,
               f"{bits} Bit ohne Blöcke: {ohne_bloecke[bits]:.1f} %",
               ha="right", fontsize=9, color=farbe)

achse.set_xscale("log", base=2)
achse.set_yscale("log")
achse.set_xticks(BLOCKGROESSEN)
achse.set_xticklabels(BLOCKGROESSEN)
achse.set_xlabel("Blockgröße (Werte je Skala)")
achse.set_ylabel("relativer Fehler (%)")
achse.set_title("Kleinere Blöcke, kleinerer Fehler")
achse.legend(frameon=False)
plt.tight_layout()
plt.show()

zeige_tabelle([{
    "Blockgröße": bg,
    "8 Bit, Fehler (%)": kurven[8][i],
    "4 Bit, Fehler (%)": kurven[4][i],
    "Skalen je Matrix": GEWICHT.size // bg,
} for i, bg in enumerate(BLOCKGROESSEN)])


In [ ]:
# ▶️ Was ein Blockformat wirklich je Gewicht kostet
def bits_je_gewicht(bits, blockgroesse, zusatz_bits=16):
    """Nominelle Bits plus die Zusatzwerte je Block, verteilt auf die Gewichte im Block."""
    return bits + zusatz_bits / blockgroesse


zeige_tabelle([{
    "Blockgröße": bg,
    "4 Bit + FP16-Skala": bits_je_gewicht(4, bg, 16),
    "4 Bit + Skala + Minimum": bits_je_gewicht(4, bg, 32),
    "8 Bit + FP16-Skala": bits_je_gewicht(8, bg, 16),
    "relativer Fehler 4 Bit (%)": kurven[4][i],
} for i, bg in enumerate(BLOCKGROESSEN)])

print("Ein asymmetrisches Verfahren legt je Block zwei Werte ab: Skala und Minimum.")
print("Das ist die Spalte in der Mitte.")


## 6 · Wirkung im Modell

Ein kleiner Gewichtsfehler ist nicht automatisch ein großer Qualitätsverlust. Deshalb vergleichen wir auch die Modellausgabe vor und nach der Quantisierung.


In [ ]:
# ▶️ Der Text, an dem gemessen wird, und die Prompts für die Beispielausgaben
MESSTEXT = (
    "CVE-2026-3224 is a critical authentication bypass in NorthGate VPN Gateway 7.2 "
    "through 7.4. An unauthenticated attacker sends a crafted SAML assertion to the "
    "/sso/acs endpoint and receives a valid administrator session. The vendor rates it "
    "9.1 on the CVSS scale, and a public proof of concept has been available since "
    "Monday. Our asset inventory lists fourteen gateways in this version range; three of "
    "them are reachable from the internet. The runbook for an internet facing "
    "authentication bypass is short. First, confirm the version on every affected host "
    "with the inventory export. Second, apply patch 7.4.3, which the vendor released on "
    "Tuesday. Third, invalidate all administrator sessions, because a session created "
    "before the patch survives the update. Fourth, search the authentication log for "
    "POST requests to /sso/acs that were answered with a redirect to the admin console. "
    "A single request from an unknown address is enough to open an incident. The change "
    "advisory board has approved an emergency window for the three exposed gateways "
    "tonight. The remaining eleven follow in the regular window on Thursday. Until the "
    "patch is applied, the mitigation is a firewall rule that blocks the endpoint at the "
    "load balancer. This breaks single sign on for remote users, so it is a trade the "
    "CISO has to approve. Report the residual risk in the morning briefing."
)

PROMPTS = [
    "Ticket: CVE-2026-3224 affects fourteen VPN gateways, three of them reachable "
    "from the internet. The first mitigation step is",
    "Incident report. At 02:14 the authentication log showed a POST request to /sso/acs "
    "from an unknown address. The analyst",
]

MESS_IDS = tokenizer(MESSTEXT, return_tensors="pt").input_ids
print(f"Messtext: {MESS_IDS.shape[1]} Tokens, {len(PROMPTS)} Prompts")


In [ ]:
# ▶️ Perplexität und Textausgabe — beides fertig
def perplexitaet(ids=MESS_IDS):
    """Exponentierter mittlerer negativer Log-Likelihood über den Messtext."""
    ausgabe = sprachmodell(ids, labels=ids)
    return float(torch.exp(ausgabe.loss))


def erzeuge(prompt, tokens=40):
    """Setzt den Prompt fort — ohne Sampling, damit der Vergleich reproduzierbar bleibt."""
    eingabe = tokenizer(prompt, return_tensors="pt")
    ausgabe = sprachmodell.generate(**eingabe, max_new_tokens=tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(ausgabe[0][eingabe.input_ids.shape[1]:],
                            skip_special_tokens=True).strip()


# Die Tensoren, die quantisiert werden: alle zweidimensionalen Gewichte in den
# Transformer-Blöcken. Einbettung und Ausgabeschicht bleiben außen vor — das ist
# in echten Formaten genauso, sie reagieren empfindlicher als die Projektionen.
GEWICHTE = [(name, tensor) for name, tensor in sprachmodell.named_parameters()
            if tensor.ndim == 2 and "embed" not in name]
ORIGINAL = {name: tensor.numpy().copy() for name, tensor in GEWICHTE}

behandelt = sum(t.numel() for _, t in GEWICHTE)
print(f"{len(GEWICHTE)} Gewichtsmatrizen mit {behandelt:,} Parametern".replace(",", "."))
print(f"das sind {behandelt / PARAMETER_GESAMT * 100:.0f} % des Modells")
print()


def schreibe_gewicht(tensor, werte):
    """Schreibt ein NumPy-Array zurück in einen Modelltensor."""
    tensor.copy_(torch.from_numpy(np.ascontiguousarray(werte, dtype=np.float32)))


def stelle_original_her():
    """Setzt alle Gewichte auf die Werte aus dem Checkpoint zurück."""
    for name, tensor in GEWICHTE:
        schreibe_gewicht(tensor, ORIGINAL[name])


print(f"Perplexität im Original: {perplexitaet():.2f}")


In [ ]:
# Vorgegebene Hilfsfunktion
def quantisiere_modell(bits, blockgroesse=None):
    """Quantisiert jedes Gewicht, rechnet es zurück und schreibt es ins Modell."""
    fehler = []
    for name, tensor in GEWICHTE:
        original = ORIGINAL[name]
        if blockgroesse is None:
            q, skala = quantisiere_absmax(original, bits)
        else:
            q, skala = quantisiere_blockweise(original, bits, blockgroesse)

        rekonstruiert = dequantisiere(q, skala).reshape(original.shape)
        schreibe_gewicht(tensor, rekonstruiert)
        fehler.append(quantisierungsfehler(original, rekonstruiert)["relativ"])

    return float(np.mean(fehler))


## 7 · Formate einordnen

GGUF, AWQ und GPTQ beschreiben unterschiedliche Quantisierungs- und Laufzeitwege. Format, Bitzahl und Runtime müssen zusammenpassen.


In [ ]:
# ▶️ Die Bits je Gewicht der GGUF-Formate, aus ihrer Blockstruktur gerechnet
def k_quant_bits(bits, block=32, skala_bits=6, minimum_bits=6,
                 superblock=256, superblock_bits=32):
    """K-Quants: quantisierte Skalen je Block, dazu FP16-Werte je Superblock."""
    return bits + (skala_bits + minimum_bits) / block + superblock_bits / superblock


FORMATE = [
    {"Format": "Q8_0", "Bits je Gewicht": bits_je_gewicht(8, 32, 16)},
    {"Format": "Q4_0", "Bits je Gewicht": bits_je_gewicht(4, 32, 16)},
    {"Format": "Q4_1", "Bits je Gewicht": bits_je_gewicht(4, 32, 32)},
    {"Format": "Q4_K", "Bits je Gewicht": k_quant_bits(4)},
    {"Format": "Q5_K", "Bits je Gewicht": k_quant_bits(5)},
    {"Format": "Q6_K", "Bits je Gewicht": k_quant_bits(6, block=16, skala_bits=8,
                                                       minimum_bits=0,
                                                       superblock_bits=16)},
]
for f in FORMATE:
    f["gegenüber 4 Bit"] = f"{f['Bits je Gewicht'] / 4:.2f}×"
    f["70B-Modell (GB)"] = speicher_gewichte(70.6e9, f["Bits je Gewicht"])

zeige_tabelle(FORMATE)

# Was das _M ausmacht: ein Teil der Tensoren eine Stufe höher. Gerechnet an den
# echten Tensorgrößen von SmolLM2-135M.
hoeher = sum(t.numel() for n, t in GEWICHTE if "v_proj" in n or "down_proj" in n)
anteil = hoeher / sum(t.numel() for _, t in GEWICHTE)
gemischt = (1 - anteil) * k_quant_bits(4) + anteil * k_quant_bits(
    6, block=16, skala_bits=8, minimum_bits=0, superblock_bits=16)

print(f"v_proj und down_proj sind {anteil * 100:.1f} % der Gewichte.")
print(f"Liegen sie in Q6_K und der Rest in Q4_K, ergibt das {gemischt:.2f} Bit je Gewicht —")
print(f"eine Mischung dieser Art meint das M in Q4_K_M.")


In [ ]:
# ▶️ Die Bausteine für den nächsten Schritt — fertig
OVERHEAD_ANTEIL = 0.15   # Aktivierungen, Puffer der Runtime, Fragmentierung
OVERHEAD_MIN = 1.0       # mindestens 1 GB, auch bei winzigen Modellen

# Die Präzisionen, sortiert von genau nach sparsam. Die Bitzahlen sind die
# gerechneten aus Abschnitt 5 und 7, nicht die nominellen.
PRAEZISIONEN = [
    ("FP16", 16.0),
    ("Q8_0", bits_je_gewicht(8, 32, 16)),
    ("Q5_K", k_quant_bits(5)),
    ("Q4_K", k_quant_bits(4)),
]


def speicher_kv_cache(modell, kontext, batch=1, bits=16):
    """Speicher für den KV-Cache in GB. Die 2 steht für Key und Value."""
    pro_token = (2 * modell["schichten"] * modell["kv_koepfe"]
                 * modell["kopf_dimension"] * bits / 8)
    return pro_token * kontext * batch / GB


print("   ".join(f"{name}: {bits:.2f} Bit" for name, bits in PRAEZISIONEN))


## Fazit

Du hast Quantisierung als kontrollierten Tausch verstanden: **weniger Speicher gegen Rekonstruktionsfehler**. Blockweise Skalierung hält diesen Fehler klein.
